<a href="https://colab.research.google.com/github/abdelhak-abdelhakem/bert-sentiment-analysis-imbd-reviews/blob/main/bert_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import

In [ ]:
import torch
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer , AutoModelForSequenceClassification , DataCollatorWithPadding , TrainingArguments , Trainer

##2. load dataset

In [ ]:
dataset = load_dataset("imdb")

##3. Tokenization

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

##4. Loading the Pre-trained Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

##5. Defining Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",           # Where to save the model
    learning_rate=2e-5,               # The initial learning rate for the AdamW optimizer.
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,                # Prevent overfitting
    eval_strategy="epoch",      # Evaluate at the end of each epoch
    save_strategy="epoch",            # Save checkpoint each epoch
    load_best_model_at_end=True,
)

## 6. Creating the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

##7. Fine-Tuning the Model

In [ ]:
trainer.train()

##8. Evaluating the Model

In [ ]:
# Evaluate the model
evaluation_results = trainer.evaluate()
print(evaluation_results)

In [ ]:
import torch

text = "I absolutely loved this movie, it was incredible!"
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = trainer.model(**inputs)

logits = outputs.logits
predicted_class_id = torch.argmax(logits, dim=1).item()

print(f"Predicted class ID: {predicted_class_id}")
print(f"Predicted label: {trainer.model.config.id2label[predicted_class_id]}")


##save the model

In [ ]:
# Save model and tokenizer
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")